In [ ]:
import os
from pathlib import Path

# Ensure relative paths resolve from the project root when running from notebooks/
if Path.cwd().name == "notebooks":
    os.chdir("..")

In [ ]:
import pandas as pd
from pathlib import Path
from klarity import parsing
from klarity import metrics
from klarity import viz

import numpy as np
import matplotlib.pyplot as plt
import math

In [ ]:
import config
from klarity import io

PLOT_DIR = config.SETTING_COMPARISON_DIR

In [ ]:
io.check_dataframes_stale()

In [ ]:
df = io.read_frame_dataframe()

In [ ]:
placements = ["placement_" + f"{i+1}" for i in range(6)]
metrics = ["mean_diameter_mm", "n_bubbles_total"]
y_labels = [rf"Pooled mean bubble diameter $\overline{{d}}$ [mm]", r"Mean bubbles per valid frame [-]"]

In [ ]:
# Effective sample sizes for autocorrelation-adjusted confidence intervals.
# Values are summed over the two independent replicates for each condition and metric.
_neff = pd.concat(
    [
        pd.read_csv(
            "data/public/temporal_independence_full_grid.csv",
            dtype={"xanthan": str},
        ),
        pd.read_csv(
            "data/public/temporal_independence_holdup.csv",
            dtype={"xanthan": str},
        ),
    ],
    ignore_index=True,
)
_neff["setting"] = (
    _neff["rpm"].astype(str) + " rpm " + _neff["lmin"].astype(str)
    + " lmin " + _neff["xanthan"].astype(str) + " xanthan"
)
_neff["placement"] = "placement_" + _neff["position"].astype(str)
_neff_sum = _neff.groupby(["placement", "setting", "metric"])["N_eff"].sum()
_METRIC_TO_TEMPORAL = {
    "n_bubbles_total": "count",
    "mean_diameter_mm": "mean_diameter_mm",
    "epsilon_obs_mid": "epsilon_obs_mid",
    "a_obs_m2_m3_mid": "a_obs_m2_m3_mid",
}
n_eff_lookup = {
    (plot_metric, setting, placement): float(v)
    for plot_metric, temp_metric in _METRIC_TO_TEMPORAL.items()
    for (placement, setting, m), v in _neff_sum.items()
    if m == temp_metric
}
print(f"N_eff lookup built: {len(n_eff_lookup)} entries")

## Condition-level estimands

Public figures use `estimand="condition"`. Mean diameter is pooled over retained
bubbles (`sum(diameter_sum_mm) / sum(diameter_count)`); count is the mean per valid
analyzed frame; and gas holdup/interfacial area are ratios of summed physical
contributions. Camera-failure black frames are excluded, while a valid analyzed frame
with zero retained detections remains a physical zero. Ratio confidence intervals use
a delete-one-frame jackknife widened by the available metric-specific N_eff. The
diameter N_eff is computed from the per-frame mean-diameter series.


## Figure grid and file naming

Every figure below sweeps **one** operating parameter across the six endoscope positions
with the other two held fixed. The legend names only the parameter that varies, so the
fixed levels live in the file name:

| stem | x axis | held fixed |
|---|---|---|
| `rpm_at_55_lmin_000_xanthan` | agitation rate | 55 L min$^{-1}$, 0.00 wt% xanthan |
| `lmin_at_100_rpm_025_xanthan` | aeration rate | 100 min$^{-1}$, 0.25 wt% xanthan |

Two markers modify the stem. Both are applied by `viz.setting_comparison_stem`, which
derives the name from the settings list actually plotted, so file name and figure content
cannot drift apart:

* prefix `holdup_area_` — gas holdup and observed-volume interfacial area density, instead of mean bubble
  diameter and bubble count;
* suffix `_all_aeration` — the complete five-setpoint aeration series including
  80 L min$^{-1}$, as opposed to the four setpoints (45/55/70/90) of the manuscript
  reference figure. Same convention as the `*_all_aeration` heat map grids in
  `create_heatmaps.ipynb`.

### Coverage

The factorial is measured in full — 4 agitation × 5 aeration × 3 xanthan = 60 settings —
and both interval lookups (`N_eff`, prolate–oblate band) cover all 360 condition–placements,
so every combination rendered here carries the same autocorrelation-adjusted CI and, where
applicable, the same depth band. Rendered:

* agitation sweeps at **all five** aeration rates, per xanthan level;
* aeration sweeps at **all four** stirrer speeds, as the complete five-setpoint series
  (`_all_aeration`);
* the four-setpoint aeration layout at 100 min$^{-1}$ only, which is the manuscript
  reference figure.

That is 30 stems per metric pair, 60 in total, each written as `.png`, `.svg` and `.pdf`.

Color assignment differs between the four- and five-setpoint aeration figures:
`viz.color_cycle` is applied in setting order, so 90 L min$^{-1}$ is pink in the
four-setpoint figures and eminence in the five-setpoint ones, with pink taken by
80 L min$^{-1}$. Do not compare the two versions by color.


In [ ]:
# --- Sweep grid ---------------------------------------------------------------------
# The factorial is measured in full: 4 agitation x 5 aeration x 3 xanthan = 60 settings.
XANTHAN_LEVELS = ["000", "0125", "025"]
RPM_LEVELS = [75, 100, 125, 150]
AER_LEVELS_ALL = [45, 55, 70, 80, 90]
AER_LEVELS_REFERENCE = [45, 55, 70, 90]  # four setpoints of the manuscript figure

# Levels the manuscript reference figures hold fixed. Everywhere else the sweeps below
# cover the whole grid, so these two only pick out which figures reproduce the manuscript.
REFERENCE_RPM = 100  # stirrer speed of the reference aeration figure
REFERENCE_AER = 55  # aeration rate of the reference agitation figure

ALL_AERATION = "_all_aeration"  # stem suffix marking the five-setpoint series


def agitation_sweep(aer_lmin: int, xanthan: str) -> list[str]:
    """Settings for an agitation sweep at one aeration rate and xanthan level."""
    return [f"{rpm} rpm {aer_lmin} lmin {xanthan} xanthan" for rpm in RPM_LEVELS]


def aeration_sweep(rpm: int, xanthan: str, aer_levels: list[int] = AER_LEVELS_ALL) -> list[str]:
    """Settings for an aeration sweep at one stirrer speed and xanthan level."""
    return [f"{rpm} rpm {aer} lmin {xanthan} xanthan" for aer in aer_levels]


# A setting absent from the pickle would otherwise become a silently empty series -- the
# figure still renders, with a gap no file name records.
_available = set(df["reactor_setting"].unique())
_wanted = {
    f"{rpm} rpm {aer} lmin {xanthan} xanthan"
    for rpm in RPM_LEVELS
    for aer in AER_LEVELS_ALL
    for xanthan in XANTHAN_LEVELS
}
assert _wanted <= _available, f"absent from frame_level_df: {sorted(_wanted - _available)}"
print(f"Grid complete: {len(_wanted)} settings present in frame_level_df")


In [ ]:
# --- Mean diameter and bubble count -------------------------------------------------
# One figure per (swept parameter, fixed level, xanthan level). `setting_comparison_stem`
# reads the fixed levels off the same settings list that is plotted, so the name cannot
# claim a condition the figure does not show. Mean diameter is pooled over retained
# bubbles; count is the arithmetic mean across valid analyzed frames.
def render_size_count(settings: list[str], suffix: str = "") -> str:
    """Render one sweep of mean diameter and bubble count; return the stem written."""
    stem = viz.setting_comparison_stem(settings, suffix=suffix)
    fig = viz.plot_settings_comparison(
        df,
        settings=settings,
        metrics=metrics,
        y_labels=y_labels,
        n_eff_lookup=n_eff_lookup,
        estimand="condition",
        outpath=PLOT_DIR / stem,
        show=False,
    )
    plt.close(fig)  # 60 figures are written here; holding them all open is what warns
    return stem


# Agitation sweeps at every aeration rate, per xanthan level.
written = [
    render_size_count(agitation_sweep(aer, xanthan))
    for aer in AER_LEVELS_ALL
    for xanthan in XANTHAN_LEVELS
]
print(f"{len(written)} agitation sweeps written")


In [ ]:
# Aeration sweeps as the complete five-setpoint series, at every stirrer speed.
written = [
    render_size_count(aeration_sweep(rpm, xanthan), suffix=ALL_AERATION)
    for rpm in RPM_LEVELS
    for xanthan in XANTHAN_LEVELS
]
print(f"{len(written)} aeration sweeps written (five setpoints)")


In [ ]:
# The four-setpoint aeration layout (45/55/70/90) of the manuscript reference figure, at
# 100 min^-1 only -- the stirrer speed it was measured at. Every other stirrer speed gets
# the complete five-setpoint series above and no four-setpoint counterpart.
written = [
    render_size_count(aeration_sweep(REFERENCE_RPM, xanthan, AER_LEVELS_REFERENCE))
    for xanthan in XANTHAN_LEVELS
]
print(f"{len(written)} reference aeration sweeps written (four setpoints, {REFERENCE_RPM} rpm)")


## Gas holdup and interfacial area

The same setting-comparison layout applied to the two derived quantities the manuscript
maps as heat maps, so every reported metric carries a confidence interval rather than
only bubble number and size.

Both are plotted as the **prolate–oblate midpoint** (`epsilon_obs_mid`,
`a_obs_m2_m3_mid`) — the quantity the heat maps shade — and the y-axis labels are taken
straight from `viz.METRIC_SPECS` so figure and colorbar cannot drift apart.

**The capped error bars are sampling uncertainty only.** The plotted point is the ratio
of summed physical contributions across valid analyzed frames; its 95% CI comes from a
delete-one-frame jackknife widened by the metric's own N_eff. The wider translucent
prolate–oblate bar is a *systematic* interval from the unobserved depth axis and does not
shrink with sample size. The two must not be read as one interval.


In [ ]:
# Gas holdup and interfacial area, labelled exactly as the manuscript heat maps
# (Figure 9 = a_obs_m2_m3_mid, S7.1 = epsilon_obs_mid).
holdup_metrics = ["epsilon_obs_mid", "a_obs_m2_m3_mid"]
holdup_y_labels = [viz.METRIC_SPECS[m]["cbar"] for m in holdup_metrics]

# --- Prolate-oblate depth band (systematic, NOT a confidence interval) ---
# These two metrics depend on the unobserved depth axis, so the reported value is the
# midpoint of the two admissible completions of the silhouette and the band is its
# interval. It is drawn behind the CI as a wider translucent bar and the two are never
# combined: the CI shrinks as frames accumulate, the band does not shrink at all.
#
# Endpoints come from headline_numbers_by_condition.csv, which is what the manuscript
# text quotes, so figure and text cannot disagree. Deliberately NOT the per-frame
# *_band_pct columns -- a different quantity (mean of the per-frame relative half-width)
# that diverges from this one by up to a factor of three in water at position 1.
_band = pd.read_csv("data/public/headline_numbers_by_condition.csv")
_band["setting"] = (
    _band["rpm_val"].astype(int).astype(str) + " rpm "
    + _band["aer_val"].astype(int).astype(str) + " lmin "
    + _band["xanthan"].astype(str)
)
_BAND_COLUMNS = {
    "epsilon_obs_mid": ("epsilon_obs_lower", "epsilon_obs_upper"),
    "a_obs_m2_m3_mid": ("a_obs_m2_m3_lower", "a_obs_m2_m3_upper"),
}
band_lookup = {
    (metric, row.setting, row.placement): (getattr(row, lo_col), getattr(row, hi_col))
    for metric, (lo_col, hi_col) in _BAND_COLUMNS.items()
    for row in _band.itertuples()
}
print(f"Band lookup built: {len(band_lookup)} (metric, setting, placement) entries")
for m, lab in zip(holdup_metrics, holdup_y_labels):
    print(f"{m:20s} -> {lab}")


In [ ]:
# Guard: assert on the drawn artists, not on the arguments passed in.
#
# The figure files are gitignored, so a silently band-less render leaves no trace in
# `git status` and is easy to ship. It has happened once already, from a klarity/viz.py
# that briefly lost the band code -- and no check *inside* viz.py can catch that case,
# because the stale viz.py has no check in it either. Hence a check here, in the caller.
#
# Each banded panel must carry two ErrorbarContainers per setting (the band and the CI),
# and the interval key legend that tells the reader which is which.
def assert_bands_drawn(fig, settings, n_metrics=len(holdup_metrics)):
    for ax in fig.axes[:n_metrics]:
        n = len(ax.containers)
        assert n == 2 * len(settings), (
            f"expected {2 * len(settings)} intervals (band + CI per setting) on "
            f"'{ax.get_ylabel()}', found {n} -- the depth band is missing"
        )
    assert fig.axes[0].get_legend() is not None, "interval key legend is missing"


def render_holdup_area(settings: list[str], suffix: str = "") -> str:
    """Render one sweep of gas holdup and interfacial area; return the stem written."""
    stem = viz.setting_comparison_stem(settings, prefix="holdup_area_", suffix=suffix)
    fig = viz.plot_settings_comparison(
        df,
        settings=settings,
        metrics=holdup_metrics,
        y_labels=holdup_y_labels,
        n_eff_lookup=n_eff_lookup,
        band_lookup=band_lookup,
        estimand="condition",
        outpath=PLOT_DIR / stem,
        show=False,
    )
    assert_bands_drawn(fig, settings)
    plt.close(fig)
    return stem


# Agitation sweeps at every aeration rate, per xanthan level.
written = [
    render_holdup_area(agitation_sweep(aer, xanthan))
    for aer in AER_LEVELS_ALL
    for xanthan in XANTHAN_LEVELS
]
print(f"{len(written)} agitation sweeps written")


In [ ]:
# Aeration sweeps: the complete five-setpoint series at every stirrer speed, plus the
# four-setpoint reference layout at 100 min^-1, matching the two cells above.
written = [
    render_holdup_area(aeration_sweep(rpm, xanthan), suffix=ALL_AERATION)
    for rpm in RPM_LEVELS
    for xanthan in XANTHAN_LEVELS
] + [
    render_holdup_area(aeration_sweep(REFERENCE_RPM, xanthan, AER_LEVELS_REFERENCE))
    for xanthan in XANTHAN_LEVELS
]
print(f"{len(written)} aeration sweeps written")


In [ ]:
# Inventory of what this notebook produced, as a check that no stem was overwritten by
# another sweep: 60 distinct stems, each in three formats.
stems = sorted({p.stem for p in PLOT_DIR.glob("*.pdf")})
print(f"{len(stems)} figure stems in {PLOT_DIR}\n")
for stem in stems:
    print(" ", stem)
